In [1]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
import matplotlib.pyplot as plt
from math import factorial

In [2]:
RESULTS_PREFIX = 'results'
HP_RESULTS_PREFIX = 'hpSearch'

In [3]:
input_dir = "/home-local2/sasah14.extra.nobkp/scratch/out/stta/sweep/bestmultiruns"
sweep_names = [
    "PACS_resnet18-BN_ERM_0", "PACS_resnet50-BN_ERM_0",
    "VLCS_resnet18-BN_ERM_0", "VLCS_resnet50-BN_ERM_0",
    "TerraIncognita_resnet18-BN_ERM_0", "TerraIncognita_resnet50-BN_ERM_0",
    "OfficeHome_resnet18-BN_ERM_0", "OfficeHome_resnet50-BN_ERM_0"
]

In [4]:
def camel_case_split(identifier):
    matches = re.finditer('.+?(?:(?<=[a-z])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])|$)', identifier)
    return [m.group(0) for m in matches]

def get_test_env(args):
    test_envs = args['test_envs']
    assert len(test_envs) == 1
    test_env = test_envs[0]
    return test_env

def get_names(df_res, test_env):
    train_out_names = [col for col in df_res.columns
                            if 'env' in col and
                                'out_acc' in col and
                                str(test_env) not in col]
    test_out_names = [col for col in df_res.columns
                            if 'env' in col and
                                'out_acc' in col and
                                str(test_env) in col]
    test_in_names = [col for col in df_res.columns
                            if 'env' in col and
                                'in_acc' in col and
                                str(test_env) in col]
    assert len(test_out_names) == 1
    assert len(test_in_names) == 1
    test_out_name = test_out_names[0]
    test_in_name = test_in_names[0]
    return train_out_names, test_out_name, test_in_name

def fill_nans(acc_dict, test_in_name, adapt_algorithm, loc_alg_name):
    # Extract loc algo setting names
    find_algo, fix_algo, granularity = camel_case_split(loc_alg_name)

    # Append algo names to dictionary
    acc_dict['adapt_algorithm'].append(adapt_algorithm)
    acc_dict['find_algo'].append(find_algo)
    acc_dict['fix_algo'].append(fix_algo)
    acc_dict['granularity'].append(granularity)

    # Append target acc to dictionary
    acc_dict[test_in_name].append(np.nan)

def fill_dg(iid_dict, orc_dict, man_dict, df, train_out_names, test_out_name, test_in_name, args):
    # Get best train and test out idx
    df['train_out_mean'] = df[train_out_names].mean(axis=1)
    best_train_out_idx = df['train_out_mean'].idxmax()
    best_test_out_idx = df[test_out_name].idxmax()

    # Get best train and test out based test-in acc
    iidacc = df[test_in_name].iloc[best_train_out_idx]
    orcacc = df[test_in_name].iloc[best_test_out_idx]

    # TODO: Assumes that DG is encountered 1st, fix it!
    # Append acc to both dictionaries
    iid_dict[test_in_name] = [iidacc]
    orc_dict[test_in_name] = [orcacc]
    man_dict[test_in_name] = []
    
    # Append algo names to both dictionaries
    iid_dict['adapt_algorithm'] = ['DG']
    iid_dict['find_algo'] = ['DG']
    iid_dict['fix_algo'] = ['DG']
    iid_dict['granularity'] = ['DG']
    orc_dict['adapt_algorithm'] = ['DG']
    orc_dict['find_algo'] = ['DG']
    orc_dict['fix_algo'] = ['DG']
    orc_dict['granularity'] = ['DG']
    man_dict['adapt_algorithm'] = []
    man_dict['find_algo'] = []
    man_dict['fix_algo'] = []
    man_dict['granularity'] = []

    # Append seed to both dictionaries
    # iid_dict['hparams_seed'].append(args['hparams_seed'])
    # iid_dict['seed'].append(args['seed'])
    # iid_dict['trial_seed'].append(args['trial_seed'])
    # orc_dict['hparams_seed'].append(args['hparams_seed'])
    # orc_dict['seed'].append(args['seed'])
    # orc_dict['trial_seed'].append(args['trial_seed'])

def fill_row_loc(acc_dict, row, test_in_name, args):
    # Append algo names to dictionary
    acc_dict['adapt_algorithm'].append(args['adapt_algorithm'])
    if 'find_algo' in row: 
        # for loc TTA
        acc_dict['find_algo'].append(args['find_algo'])
        acc_dict['fix_algo'].append(args['fix_algo'])
        acc_dict['granularity'].append(args['granularity'])
    else:
        # for full TTA
        acc_dict['find_algo'].append('None')
        acc_dict['fix_algo'].append('None')
        acc_dict['granularity'].append('None')

    # Append target acc to dictionary
    acc_dict[test_in_name].append(row[test_in_name])

def fill_meta_data(acc_dict, test_in_name, dataset, backbone, dg_name, mark, hp_mode):
    acc_dict['hp_mode'] = [hp_mode for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['dataset'] = [dataset for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['backbone'] = [backbone for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['dg'] = [dg_name for _ in range(len(acc_dict[test_in_name]))]
    acc_dict['mark'] = [mark for _ in range(len(acc_dict[test_in_name]))]

def get_df(acc_dict):
    # Create dataframe from dictionary
    df = pd.DataFrame.from_dict(acc_dict)

    # Compute average accuracy
    df['Avg'] = df[[col for col in df.columns
                            if 'in_acc' in col]].mean(axis=1)

    # Sort dataframe columns
    df = df[['adapt_algorithm', 'granularity', 'find_algo', 'fix_algo',
                     'env0_in_acc', 'env1_in_acc', 'env2_in_acc', 'env3_in_acc', 'Avg',
                     'hp_mode', 'dataset', 'backbone', 'dg', 'mark']]
    
    return df

def calculate_shapley_values(powerset):
    """
    Calculate Shapley values of players in a cooperative game.
    
    The Shapley value is a solution concept in cooperative game theory that assigns a payout to 
    each player in the game, proportional to their marginal contribution to the total payout.

    Parameters:
    powerset (dict): a dictionary with keys as tuples representing subsets of players and 
    values as their corresponding game values.

    Returns:
    dict: a dictionary with players as keys and their corresponding Shapley values as values.
    """

    # Ensure the input is a dictionary and keys are tuples
    assert isinstance(powerset, dict), "Input should be a dictionary"
    assert all(isinstance(i, tuple) for i in powerset.keys()), "Keys should be tuples"

    # Flatten the list of subsets to find all unique players
    players = set(player for subset in powerset.keys() for player in subset)
    total_players = len(players)  # total number of players

    # Initialize the Shapley values with 0 for all players
    shapley_values = {player: 0 for player in players}

    # Iterate over each subset in the power set
    for subset, subset_value in powerset.items():
        players_in_subset = len(subset)  # number of players in the subset

        # Iterate over each player in the subset
        for player in subset:
            # Create the subset without the current player
            subset_excluding_player = tuple([p for p in subset if p != player])
            # Get the value for the subset without the player or 0 if it doesn't exist
            value_of_excluded_subset = powerset.get(subset_excluding_player, 0)
            
            # Calculate the player's marginal contribution
            marginal_contribution = subset_value - value_of_excluded_subset
                
            # Compute the Shapley value for the player
            weight = factorial(players_in_subset - 1) * factorial(total_players - players_in_subset) / factorial(total_players)
            shapley_values[player] += marginal_contribution * weight

    return shapley_values

# powerset = {
#     (): 0,
#     ('A',): 1,
#     ('B',): 2,
#     ('C',): 3,
#     ('A', 'B'): 4,
#     ('A', 'C'): 5,
#     ('B', 'C'): 7,
#     ('A', 'B', 'C'): 10
# }

def get_kp_values(df, all_box_list, target_col):
    # Store accs
    df['tta_acc'] = df[target_col]

    #########################################
    # Get correlation and shapley values code
    #########################################

    # Get dataframe with only relevant columns
    df_inq = df[['find_algo'] + all_box_list + ['tta_acc']]

    # Compute pearson correlation values
    corr_values = df_inq.corr()['tta_acc'].drop('tta_acc').to_dict()
    corr_values = dict(sorted(corr_values.items(), key=lambda x: x))
    corr_values['classifier'] = corr_values.pop('classifier')

    # Compute shapley values
    powerset_dict = {tuple(row['find_algo'].split('_')) : row['tta_acc'] for idx, row in df_inq.iterrows()}
    powerset_dict[()] = powerset_dict.pop(tuple(['none']))
    shapley_values = calculate_shapley_values(powerset_dict)
    shapley_values = dict(sorted(shapley_values.items(), key=lambda x: x))
    shapley_values['classifier'] = shapley_values.pop('classifier')

    # Add to df
    df_corr = pd.DataFrame.from_dict(corr_values, orient='index').rename(columns={0: 'corr'})
    df_shap = pd.DataFrame.from_dict(shapley_values, orient='index').rename(columns={0: 'shap'})


    ############################
    # Get max weight code
    ############################
    # Get dataframe with only relevant columns
    df_inq = df[['find_algo', 'count'] + all_box_list + ['tta_acc']]

    # Compute greedy shapley values
    full_name = ''
    greedy_rank = ['none']
    for layer_name in df_shap.sort_values(by='shap', ascending=False).index:
        if full_name != '':
            full_name += '_'
        full_name += layer_name
        greedy_rank.append(full_name)
    greedy_rank = ['_'.join(sorted(sorted(find_algo.split('_')), key=len)) for find_algo in greedy_rank]

    # Add greedy shapley values to grouped data
    greedy_shap_values = [df_inq[df_inq['find_algo'] == find_algo]['tta_acc'].values[0] for find_algo in greedy_rank]

    # Compute greedy correlation values
    full_name = ''
    greedy_rank = ['none']
    for layer_name in df_corr.sort_values(by='corr', ascending=False).index:
        if full_name != '':
            full_name += '_'
        full_name += layer_name
        greedy_rank.append(full_name)
    greedy_rank = ['_'.join(sorted(sorted(find_algo.split('_')), key=len)) for find_algo in greedy_rank]

    # Add greedy correlation values to grouped data
    greedy_corr_values = [df_inq[df_inq['find_algo'] == find_algo]['tta_acc'].values[0] for find_algo in greedy_rank]

    # Compute greedy best single values
    full_name = ''
    greedy_rank = ['none']
    for layer_name in df_inq[df_inq['count'] == 1].sort_values(by='tta_acc', ascending=False)['find_algo'].values:
        if full_name != '':
            full_name += '_'
        full_name += layer_name
        greedy_rank.append(full_name)
    greedy_rank = ['_'.join(sorted(sorted(find_algo.split('_')), key=len)) for find_algo in greedy_rank]

    ############################
    # Add greedy best single values to grouped data
    greedy_sing_values = [df_inq[df_inq['find_algo'] == find_algo]['tta_acc'].values[0] for find_algo in greedy_rank]

    # Get best shapley, correlation and single values
    shapBinaryBest_GreedyKP = max(greedy_shap_values)
    corrBinaryBest_GreedyKP = max(greedy_corr_values)
    singBinaryBest_GreedyKP = max(greedy_sing_values)

    # Group by count and compute min, mean, max
    grouped_data = df_inq.groupby('count')['tta_acc'].agg(['min', 'mean', 'max']).reset_index()

    # Extract the data to be fit
    xdata, ydata = grouped_data['count'], grouped_data['max']

    # Fit a 2-degree polynomial
    coefficients = np.polyfit(xdata, ydata, 2)
    mu_fit = -coefficients[1] / (2 * coefficients[0])

    max2d_wt = int(mu_fit.round())
    if max2d_wt < 0:
        max2d_wt = 0
    elif max2d_wt > 6:
        max2d_wt = 6
    shapBinaryMax2d_GreedyKP = greedy_shap_values[max2d_wt]
    corrBinaryMax2d_GreedyKP = greedy_corr_values[max2d_wt]
    singBinaryMax2d_GreedyKP = greedy_sing_values[max2d_wt]
    bestBinaryMax2d_GreedyKP = df_inq[df_inq['count'] == max2d_wt]['tta_acc'].max()

    return shapBinaryBest_GreedyKP, corrBinaryBest_GreedyKP, singBinaryBest_GreedyKP, \
            shapBinaryMax2d_GreedyKP, corrBinaryMax2d_GreedyKP, singBinaryMax2d_GreedyKP, \
            max2d_wt, bestBinaryMax2d_GreedyKP

In [5]:
sweep_name = sweep_names[0]
print(sweep_name)
print(sweep_names)
print(len(sweep_names))

PACS_resnet18-BN_ERM_0
['PACS_resnet18-BN_ERM_0', 'PACS_resnet50-BN_ERM_0', 'VLCS_resnet18-BN_ERM_0', 'VLCS_resnet50-BN_ERM_0', 'TerraIncognita_resnet18-BN_ERM_0', 'TerraIncognita_resnet50-BN_ERM_0', 'OfficeHome_resnet18-BN_ERM_0', 'OfficeHome_resnet50-BN_ERM_0']
8


In [6]:
# Get sweep dir
sweep_dir = os.path.join(input_dir, sweep_name)

# Get runs
runs = [run for run in sorted(os.listdir(sweep_dir))
                                        if os.path.isdir(os.path.join(sweep_dir, run))]

# Get unique list of all results fnames
res_fnames = sorted(list(set([fname for run in runs
                                        for fname in os.listdir(os.path.join(sweep_dir, run))
                                            if RESULTS_PREFIX in fname and\
                                                'hp' not in fname and\
                                                '.jsonl' in fname])))

# Get unique list of all hp results fnames
man_fnames = sorted(list(set([fname for run in runs
                                        for fname in os.listdir(os.path.join(sweep_dir, run))
                                            if HP_RESULTS_PREFIX in fname and\
                                                'Man' in fname and\
                                                '.jsonl' in fname])))
iid_fnames = sorted(list(set([fname for run in runs
                                        for fname in os.listdir(os.path.join(sweep_dir, run))
                                            if HP_RESULTS_PREFIX in fname and\
                                                'Iid' in fname and\
                                                '.jsonl' in fname])))
orc_fnames = sorted(list(set([fname for run in runs
                                        for fname in os.listdir(os.path.join(sweep_dir, run))
                                            if HP_RESULTS_PREFIX in fname and\
                                                'Orc' in fname and\
                                                '.jsonl' in fname])))
res_fnames, man_fnames

(['results_PseudoLabel_BestmultiTrainsubsetBlock.jsonl',
  'results_SHOT_BestmultiTrainsubsetBlock.jsonl',
  'results_TentFull_BestmultiTrainsubsetBlock.jsonl'],
 ['hpSearch_PseudoLabel_BestmultiTrainsubsetBlock_Man.jsonl',
  'hpSearch_SHOT_BestmultiTrainsubsetBlock_Man.jsonl',
  'hpSearch_TentFull_BestmultiTrainsubsetBlock_Man.jsonl'])

# Full analysis

In [7]:
# Fixed params
full_data_col = 'hp_acc'
low_data_fracs = [0, 0.25, 0.5, 0.75, 1.0]
sweep_tables = {}

# For different datasets and backbones
for sweep_name in sweep_names:
    dataset, backbone, dg_name, mark = sweep_name.split('_')
    print('dataset: {}, backbone: {}'.format(dataset, backbone))

    # Get sweep dir
    sweep_dir = os.path.join(input_dir, sweep_name)

    # man_dict = {}
    man_dict = {'adapt_algorithm': [], 'find_algo': [], 'fix_algo': [], 'granularity': [],
                'env0_in_acc': [], 'env1_in_acc': [], 'env2_in_acc': [], 'env3_in_acc': [],
                'hp_mode': [], 'dataset': [], 'backbone': [], 'dg': [], 'mark': []}

    # Get runs
    runs = [run for run in sorted(os.listdir(sweep_dir))
                                        if os.path.isdir(os.path.join(sweep_dir, run))]

    # For different TTAs
    # For different envs per dataset and backbone
    # run_to_testinname = {}  # dict to map a given run to test-in name
    
    for run in runs:
        for f_idx, (res_fname, man_fname) in enumerate(zip(res_fnames, man_fnames)):
            tta_name = man_fname[:-6].split('_')[1]
            print('\ttta: {}'.format(tta_name))
        
            run_dir = os.path.join(sweep_dir, run)

            # Read in results
            df_res = pd.read_json(os.path.join(run_dir, res_fname), lines=True)
            # Get args
            args = df_res['args'][0]
            # Get test env
            test_env = get_test_env(args)
            print('\t\ttest_env: {}'.format(test_env))

            # Get train and test in and out names
            train_out_names, test_out_name, test_in_name = get_names(df_res, test_env)

            # Read in Man results
            df_man = pd.read_json(os.path.join(run_dir, man_fname), lines=True)

            # Get low data cols
            # low_data_cols = [col for col in df_man.columns.values if 'step' in col]
            low_data_cols = ['hp_acc']
            low_data_fracs = [1.0]

            # For different low data setups
            for target_col, low_data_frac in zip(low_data_cols, low_data_fracs):
                # get best rows for each find_algo column
                print('\t\t\ttarget_col: {}'.format(target_col))
                df_man_best = df_man.sort_values(by=[target_col], ascending=False).drop_duplicates(subset='find_algo', keep='first')

                # Compute few stats
                full_box_name = max(df_man_best['find_algo'], key=len)
                all_box_list = full_box_name.split('_')
                single_box_flag = np.array([False]*df_man_best.shape[0])
                train_count_arr = np.zeros(df_man_best.shape[0])
                for box in all_box_list:
                    single_box_flag = single_box_flag | (df_man_best['find_algo'] == box).values
                    df_man_best[box] = [box in row['find_algo'] for idx, row in df_man_best.iterrows()]
                    train_count_arr += df_man_best[box].astype(int).values
                df_man_best['count'] = train_count_arr.astype(int)

                only_box_list = all_box_list[1:-1]
                train_frac_arr = np.zeros(df_man_best.shape[0])
                for box in only_box_list:
                    train_frac_arr += df_man_best[box].astype(int).values
                df_man_best['train_frac'] = train_frac_arr.astype(int) / 4.0

                # get accs different loc TTA
                dg_acc = df_man_best[df_man_best['find_algo'] == 'none'][target_col].values[0]
                full_acc = df_man_best[df_man_best['find_algo'] == full_box_name][target_col].values[0]
                best_powerset_acc = df_man_best[target_col].values[0]
                worst_powerset_acc = df_man_best[target_col].values[-1]
                best_single_acc = df_man_best[single_box_flag][target_col].values[0]
                df_man_best['loc>full'] = df_man_best[target_col] - full_acc
                df_man_best['subset_quality'] = df_man_best[target_col] - df_man_best[target_col].iloc[-1]

                # get accs different KP TTA
                shapBinaryBest_GreedyKP, corrBinaryBest_GreedyKP, singBinaryBest_GreedyKP, \
                    shapBinaryMax2d_GreedyKP, corrBinaryMax2d_GreedyKP, singBinaryMax2d_GreedyKP, \
                    max2d_wt, bestBinaryMax2d_GreedyKP = get_kp_values(df_man_best, all_box_list, target_col)

                # Get best train and test out idx
                man_dict[test_in_name].append(dg_acc)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append('DG')
                    man_dict['find_algo'].append('DG')
                    man_dict['fix_algo'].append('DG')
                    man_dict['granularity'].append('DG')
                
                man_dict[test_in_name].append(full_acc)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('Full')
                    man_dict['fix_algo'].append('None')
                    man_dict['granularity'].append('None')

                man_dict[test_in_name].append(best_single_acc)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('Best single')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(best_powerset_acc)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('Best powerset')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(worst_powerset_acc)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('Worst powerset')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(shapBinaryBest_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('shapBinaryBest_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(corrBinaryBest_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('corrBinaryBest_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(singBinaryBest_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('singBinaryBest_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(shapBinaryMax2d_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('shapBinaryMax2d_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(corrBinaryMax2d_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('corrBinaryMax2d_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(singBinaryMax2d_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('singBinaryMax2d_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                man_dict[test_in_name].append(bestBinaryMax2d_GreedyKP)
                if test_in_name == 'env0_in_acc':
                    man_dict['adapt_algorithm'].append(tta_name)
                    man_dict['find_algo'].append('bestBinaryMax2d_GreedyKP')
                    man_dict['fix_algo'].append('Trainsubset')
                    man_dict['granularity'].append('Block')

                # Print results
                if True:
                    print("\t\t\t\tBest powerset:\t\t", best_powerset_acc)
                    print("\t\t\t\tBest shapBinary_GreedyKP:\t", shapBinaryBest_GreedyKP)
                    print("\t\t\t\tBest corrBinary_GreedyKP:\t", corrBinaryBest_GreedyKP)
                    print("\t\t\t\tBest singBinary_GreedyKP:\t", singBinaryBest_GreedyKP)
                    print("\t\t\t\tmax2d_wt:\t", max2d_wt)
                    print("\t\t\t\tBest shapBinaryMax2d_GreedyKP:\t", shapBinaryMax2d_GreedyKP)
                    print("\t\t\t\tBest corrBinaryMax2d_GreedyKP:\t", corrBinaryMax2d_GreedyKP)
                    print("\t\t\t\tBest singBinaryMax2d_GreedyKP:\t", singBinaryMax2d_GreedyKP)
                    print("\t\t\t\tBest bestBinaryMax2d_GreedyKP:\t", bestBinaryMax2d_GreedyKP)
                    print("\t\t\t\tWorst powerset:\t\t", worst_powerset_acc)
                    print("\t\t\t\tDG:\t\t\t", dg_acc)
                    print("\t\t\t\tFull TTA:\t\t", full_acc)
                    print('\t\t\t\tBest single block:\t', best_single_acc)

                    print("\t\t\t\t" + "*"*60)
                    print("\t\t\t\tbest powerset > best single: \t", (best_powerset_acc - best_single_acc))
                    print("\t\t\t\tbest powerset > full:\t\t", (best_powerset_acc - full_acc))
                    print("\t\t\t\tbest powerset > dg:\t\t", (best_powerset_acc - dg_acc))
                    print("\t\t\t\tbest powerset > worst powerset:\t", (best_powerset_acc - worst_powerset_acc))
                    print("\t\t\t\t" + "*"*60)

    # Check if dictionary is empty
    if len(man_dict) == 0:
        print('Warning: Empty dictionaries created for {}'.format(sweep_name))
        continue

    # Fill meta data
    fill_meta_data(man_dict, test_in_name, dataset, backbone, dg_name, mark, 'Man')

    # Create dataframe from dictionary
    df_man = get_df(man_dict)

    # Save dataframe to csv
    df_man.to_csv(os.path.join(sweep_dir, sweep_name + '_man2.csv'), index=False)

    # Add dataframe to sweep tables dictionary
    sweep_tables[sweep_name + '_man2'] = df_man

# Check if sweep_tables is empty or contains single dataframe
if len(sweep_tables) == 0:
    print('Warning: Empty sweep_tables created! No results found!')
elif len(sweep_tables) == 2:
    print('Only single dataframe found in sweep_tables! No merging needed!')

    # Save merged dataframes to csv
    df_man.to_csv(os.path.join(input_dir, 'man2.csv'), index=False)
else:
    print('Merging dataframes...')
    # Merge all dataframes in sweep_tables
    df_all_man = pd.concat([df for dfname, df in sweep_tables.items() if 'man2' in dfname])

    # Save merged dataframes to csv
    df_all_man.to_csv(os.path.join(input_dir, 'man2.csv'), index=False)

dataset: PACS, backbone: resnet18-BN
	tta: PseudoLabel
		test_env: 1
			target_col: hp_acc
				Best powerset:		 0.8134328358208951
				Worst powerset:		 0.742004264392324
				DG:			 0.760127931769722
				Full TTA:		 0.784648187633262
				Best single block:	 0.797974413646055
				************************************************************
				best powerset > best single: 	 0.015458422174840059
				best powerset > full:		 0.028784648187633044
				best powerset > dg:		 0.05330490405117305
				best powerset > worst powerset:	 0.07142857142857106
				************************************************************
	tta: SHOT
		test_env: 1
			target_col: hp_acc
				Best powerset:		 0.8192963752665241
				Worst powerset:		 0.760127931769722
				DG:			 0.760127931769722
				Full TTA:		 0.79957356076759
				Best single block:	 0.8070362473347541
				************************************************************
				best powerset > best single: 	 0.012260127931770004
				best powerset > full:		 0.019

In [ ]:
man_dict.keys()

In [ ]:
{k:len(v) for k, v in man_dict.items()}

In [ ]:
man_dict['adapt_algorithm']

In [ ]:
man_dict['find_algo']

In [ ]:
man_dict['dataset']

In [ ]:
runs = [run for run in sorted(os.listdir(sweep_dir))
                                        if os.path.isdir(os.path.join(sweep_dir, run))]

In [ ]:
runs

In [ ]:
df_all_man

In [ ]:
# For different datasets and backbones
sweep_start_idx = 0
sweep_end_idx = 8

# For different TTAs
man_start_idx = 0
man_end_idx = 3

# For different envs per dataset and backbone
run_start_idx = 0
run_end_idx = 4


In [ ]:
# Fixed params
full_data_col = 'hp_acc'
low_data_fracs = [0, 0.25, 0.5, 0.75, 1.0]

# For different datasets and backbones
for sweep_name in sweep_names[sweep_start_idx:sweep_end_idx]:
    dataset, backbone, dg, mark = sweep_name.split('_')
    print('dataset: {}, backbone: {}'.format(dataset, backbone))

    # Get sweep dir
    sweep_dir = os.path.join(input_dir, sweep_name)

    # Get runs
    runs = [run for run in sorted(os.listdir(sweep_dir))
                                        if os.path.isdir(os.path.join(sweep_dir, run))]

    # For different TTAs
    for res_fname, man_fname in zip(res_fnames[man_start_idx:man_end_idx], man_fnames[man_start_idx:man_end_idx]):
        tta_name = man_fname[:-6].split('_')[1]
        print('\ttta: {}'.format(tta_name))

        # For different envs per dataset and backbone
        for run in runs[run_start_idx:run_end_idx]:
            run_dir = os.path.join(sweep_dir, run)

            # Read in results
            df_res = pd.read_json(os.path.join(run_dir, res_fname), lines=True)
            # Get args
            args = df_res['args'][0]
            # Get test env
            test_env = get_test_env(args)
            print('\t\ttest_env: {}'.format(test_env))

            # Read in Man results
            df_man = pd.read_json(os.path.join(run_dir, man_fname), lines=True)

            # Get low data cols
            # low_data_cols = [col for col in df_man.columns.values if 'step' in col]
            low_data_cols = ['hp_acc']
            low_data_fracs = [1.0]

            # For different low data setups
            for target_col, low_data_frac in zip(low_data_cols, low_data_fracs):
                # get best rows for each find_algo column
                print('\t\t\ttarget_col: {}'.format(target_col))
                df_man_best = df_man.sort_values(by=[target_col], ascending=False).drop_duplicates(subset='find_algo', keep='first')

                # Compute few stats
                full_box_name = max(df_man_best['find_algo'], key=len)
                all_box_list = full_box_name.split('_')
                single_box_flag = np.array([False]*df_man_best.shape[0])
                for box in all_box_list:
                    single_box_flag = single_box_flag | (df_man_best['find_algo'] == box).values
                    df_man_best[box] = [box in row['find_algo'] for idx, row in df_man_best.iterrows()]

                only_box_list = all_box_list[1:-1]
                train_frac_arr = np.zeros(df_man_best.shape[0])
                for box in only_box_list:
                    train_frac_arr += df_man_best[box].astype(int).values
                df_man_best['train_frac'] = train_frac_arr.astype(int) / 4.0

                # get accs different loc TTA
                dg_acc = df_man_best[df_man_best['find_algo'] == 'none'][target_col].values[0]
                full_acc = df_man_best[df_man_best['find_algo'] == full_box_name][target_col].values[0]
                best_powerset_acc = df_man_best[target_col].values[0]
                worst_powerset_acc = df_man_best[target_col].values[-1]
                best_single_acc = df_man_best[single_box_flag][target_col].values[0]
                df_man_best['loc>full'] = df_man_best[target_col] - full_acc
                df_man_best['subset_quality'] = df_man_best[target_col] - df_man_best[target_col].iloc[-1]

                # Print results
                if True:
                    print("\t\t\t\tBest powerset:\t\t", best_powerset_acc)
                    print("\t\t\t\tWorst powerset:\t\t", worst_powerset_acc)
                    print("\t\t\t\tDG:\t\t\t", dg_acc)
                    print("\t\t\t\tFull TTA:\t\t", full_acc)
                    print('\t\t\t\tBest single block:\t', best_single_acc)

                    print("\t\t\t\t" + "*"*60)
                    print("\t\t\t\tbest powerset > best single: \t", (best_powerset_acc - best_single_acc))
                    print("\t\t\t\tbest powerset > full:\t\t", (best_powerset_acc - full_acc))
                    print("\t\t\t\tbest powerset > dg:\t\t", (best_powerset_acc - dg_acc))
                    print("\t\t\t\tbest powerset > worst powerset:\t", (best_powerset_acc - worst_powerset_acc))
                    print("\t\t\t\t" + "*"*60)

                # Plot results
                if True:
                    fig, axes = plt.subplots(nrows=1, ncols=6, figsize=(18, 3))
                    # fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(9, 3))
                    ax_idx = 0

                    df_man_best[single_box_flag][['find_algo', 'loc>full']].sort_values(by='find_algo', ascending=True).set_index('find_algo').loc[only_box_list].plot.line(ax=axes[ax_idx], title='Single block vs full')
                    axes[ax_idx].axhline(0, color='black', linestyle='--')
                    ax_idx += 1
                    
                    df_man_best[only_box_list + [target_col]].corr()[target_col].drop(target_col).plot.line(ax=axes[ax_idx], title='Corr, powerset: Block vs {}'.format(target_col))
                    axes[ax_idx].axhline(0, color='black', linestyle='--')
                    ax_idx += 1
                    
                    # For different trainable fractions
                    for idx, train_fraction in enumerate(sorted(df_man_best['train_frac'].unique())[1:-1]):
                        mask_flag = (df_man_best['train_frac'] == train_fraction).values & (~ df_man_best['classifier'].values) & (~ df_man_best['layer0'].values)
                        temp_df = df_man_best[mask_flag]
                        # print(temp_df[['find_algo', 'train_frac'] + [target_col]])
                        temp_df[only_box_list + [target_col]].corr()[target_col].drop(target_col).plot.line(ax=axes[ax_idx], title=f'Corr, frac={train_fraction}: Block vs {target_col}')
                        axes[ax_idx].axhline(0, color='black', linestyle='--')
                        ax_idx += 1
                    # assert False

                    df_man_best[['train_frac'] + [target_col]].corr()[target_col].drop(target_col).plot.bar(ax=axes[ax_idx], title=f'Corr: All metrics vs {target_col}')
                    axes[ax_idx].axhline(0, color='black', linestyle='--')
                    # df_man_best[['train_frac', 'subset_quality'] + [target_col]].corr()[target_col].drop(target_col).plot.bar(ax=axes[ax_idx], title=f'Corr: All metrics vs {target_col}')
                    ax_idx += 1
                    
                    plt.tight_layout()
                    plt.show()

                # Plot results
                if False:
                    # x_axis = np.array(list(range(df_man_best.shape[0])))
                    plt.figure(figsize=(15, 5))
                    plt.plot(df_man_best['find_algo'], df_man_best[target_col], label=target_col)
                    plt.xticks(rotation=90)
                    plt.title(f'Full plot')
                    plt.legend()
                    plt.show()
                # assert False

            # break
        # break
    # break

            # # get best rows for each find_algo column
            # for 
            # df_man_best = df_man.sort_values(by=[target_col], ascending=False).drop_duplicates(subset='find_algo', keep='first')

## Set params

In [ ]:
# different data size``
# target_col = 'env0_in_step10_acc'
# target_col = 'env0_in_step20_acc'
# target_col = 'env0_in_step31_acc'
# target_col = 'env0_in_step41_acc'
target_col = 'hp_acc'

# different TTA
man_fname = man_fnames[0]
# man_fname = man_fnames[1]
# man_fname = man_fnames[2]

# different envs
# run = runs[0]
# run = runs[1]
run = runs[2]
# run = runs[3]
print(man_fname, run)

In [ ]:
# Iterate over results fnames
res_fname = res_fnames[0]

# Get run dir and results path
run_dir = os.path.join(sweep_dir, run)
res_path = os.path.join(run_dir, res_fname)

# print('Results file {}'.format(res_path))

# Read in results
df_res = pd.read_json(os.path.join(run_dir, res_fname), lines=True)

# Get args
args = df_res['args'][0]

# Get test env
test_env = get_test_env(args)

# Get train and test in and out names
train_out_names, test_out_name, test_in_name = get_names(df_res, test_env)

In [ ]:
# Get setup names
dataset, backbone, dg, mark = sweep_name.split('_')
tta_name = man_fname[:-6].split('_')[1]

# Read in Man results
df_man = pd.read_json(os.path.join(run_dir, man_fname), lines=True)

# get best rows for each find_algo column
df_man_best = df_man.sort_values(by=[target_col], ascending=False).drop_duplicates(subset='find_algo', keep='first')

In [ ]:
# Compute few stats
full_box_name = max(df_man_best['find_algo'], key=len)
all_box_list = full_box_name.split('_')
single_box_flag = np.array([False]*df_man_best.shape[0])
for box in all_box_list:
    single_box_flag = single_box_flag | (df_man_best['find_algo'] == box).values
    df_man_best[box] = [box in row['find_algo'] for idx, row in df_man_best.iterrows()]

only_box_list = all_box_list[1:-1]
train_frac_arr = np.zeros(df_man_best.shape[0])
for box in only_box_list:
    train_frac_arr += df_man_best[box].astype(int).values
df_man_best['train_frac'] = train_frac_arr.astype(int)

In [ ]:
# get accs different loc TTA
dg_acc = df_man_best[df_man_best['find_algo'] == 'none'][target_col].values[0]
full_acc = df_man_best[df_man_best['find_algo'] == full_box_name][target_col].values[0]
best_powerset_acc = df_man_best[target_col].values[0]
worst_powerset_acc = df_man_best[target_col].values[-1]
best_single_acc = df_man_best[single_box_flag][target_col].values[0]
df_man_best['loc>full'] = df_man_best[target_col] - full_acc
df_man_best['subset_quality'] = df_man_best[target_col] - df_man_best[target_col].iloc[-1]

# Plots
# df_man_best[single_box_flag][['find_algo', 'loc>full']].sort_values(by='find_algo', ascending=True).set_index('find_algo').loc[all_box_list].plot.bar()
# df_man_best[all_box_list + [target_col]].corr()[target_col].drop(target_col).plot.bar()

In [ ]:
print(dataset, backbone, tta_name)
print("Best powerset%:\t\t", best_powerset_acc*100)
print("Worst powerset%:\t", worst_powerset_acc*100)
print("DG%:\t\t\t", dg_acc*100)
print("Full TTA%:\t\t", full_acc*100)
print('Best single block%:\t', best_single_acc*100)

print("*"*60)
print("best powerset > best single%: \t", (best_powerset_acc - best_single_acc)*100)
print("best powerset > full%:\t\t", (best_powerset_acc - full_acc)*100)
print("best powerset > dg%:\t\t", (best_powerset_acc - dg_acc)*100)
print("best powerset > worst powerset%:", (best_powerset_acc - worst_powerset_acc)*100)
print("*"*60)

In [ ]:
plt.figure()
df_man_best[single_box_flag][['find_algo', 'loc>full']].sort_values(by='find_algo', ascending=True).set_index('find_algo').loc[all_box_list].plot.bar()
plt.title(f'Single block plot')
plt.show()

plt.figure()
plt.title(f'Correlation plot')
df_man_best[all_box_list + [target_col]].corr()[target_col].drop(target_col).plot.bar()
plt.show()

plt.figure()
plt.title(f'What matters plot')
df_man_best[['train_frac', 'subset_quality'] + [target_col]].corr()[target_col].drop(target_col).plot.bar()
plt.show()

In [ ]:
# x_axis = np.array(list(range(df_man_best.shape[0])))
plt.figure(figsize=(20, 10))
plt.plot(df_man_best['find_algo'], df_man_best[target_col], label=target_col)
plt.xticks(rotation=90)
plt.title(f'Full plot')
plt.legend()
plt.show()

# plt.figure()
# plt.plot(df_man_best['find_algo'].iloc[:20], df_man_best[target_col].iloc[:20], label=target_col)
# plt.legend()
# plt.xticks(rotation=90)
# plt.title(f'best 20')
# plt.show()

# plt.figure()
# plt.plot(df_man_best['find_algo'].iloc[-20:], df_man_best[target_col].iloc[-20:], label=target_col)
# plt.legend()
# plt.xticks(rotation=90)
# plt.title(f'worst 20')
# plt.show()

In [ ]:
df_man_best[['train_frac'] + [target_col]]

In [ ]:
for train_fraction in sorted(df_man_best['train_frac'].unique())[1:-1]:
    temp_df = df_man_best[df_man_best['train_frac'] == train_fraction]
    plt.figure()
    plt.title(f'Correlation plot with train_fraction={train_fraction/4}')
    temp_df[only_box_list + [target_col]].corr()[target_col].drop(target_col).plot.bar()

In [ ]:
df_man_best.columns

In [ ]:
# df_man_best[single_box_flag][['find_algo', 'loc>full']].sort_values(by='find_algo', ascending=True).set_index('find_algo').loc[all_box_list]

In [ ]:
# x_axis = np.array(list(range(df_man_best.shape[0])))
# for box in all_box_list:
#     plt.figure()
#     plt.plot(x_axis, df_man_best[target_col].values, label='hp_acc')
#     box_bool = (df_man_best[box].astype(int).values * (best_powerset_acc -  worst_powerset_acc)) + worst_powerset_acc
#     plt.scatter(x_axis, box_bool, label=box)
#     plt.legend()